In [1]:
import os
import cv2
import random
import pickle
import numpy as np
from tqdm import tqdm
import mediapipe as mp
import matplotlib.pyplot as plt
from scipy.signal import medfilt
from collections import defaultdict
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

2025-03-26 01:13:28.838750: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-26 01:13:28.852007: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742969608.865528  196680 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742969608.868587  196680 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-26 01:13:28.880856: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
class Config():
    def __init__(self):        
        self.frame_l = 64 # the length of frames
        self.joint_n = 33 # the number of joints
        self.joint_d = 3 # the dimension of classes        
        self.save_dir = r'skeletized'
C = Config()

In [3]:
def process_video(video_path, holistic, output_folder):
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Failed to open video file {video_path}")
        return

    # Initialize an empty list to store frame coordinates
    video_coords = []

    # Process each frame in the video
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Flip the frame horizontally for a mirrored view
        frame = cv2.flip(frame, 1)

        # Convert the BGR image to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Process the frame with MediaPipe Holistic
        results = holistic.process(rgb_frame)

        # Extract and store the coordinate data
        if results.pose_landmarks:
            frame_coords = []
            for landmark in results.pose_landmarks.landmark:
                # Store the normalized coordinates (x, y, z) as a list
                frame_coords.append([landmark.x, landmark.y, landmark.z])
            # Append the coordinates of this frame to the video_coords list
            video_coords.append(frame_coords)

    # Convert the list of frame coordinates into a numpy array
    video_coords_array = np.array(video_coords)

    # Save the numpy array of frame coordinates
    filename = os.path.splitext(os.path.basename(video_path))[0] + ".npy"
    output_path = os.path.join(output_folder, filename)
    np.save(output_path, video_coords_array)

    # Release resources
    cap.release()

def process_video_clips(input_folder, output_info_file, output_folder):
    # Initialize MediaPipe Holistic
    mp_holistic = mp.solutions.holistic
    holistic = mp_holistic.Holistic(static_image_mode=False)

    # Create or open the output info file
    with open(output_info_file, "w") as info_file:
        #info_file.write("Serial Number\tAction\tFilename\n")

        # Process each video clip in the input folder
        serial_number = 1
        for filename in os.listdir(input_folder):
            if filename.endswith(".mp4"):
                video_path = os.path.join(input_folder, filename)
                # Extract action name from video file name
                action_name = filename.split("_")[1]  # Assuming action name is after the first underscore
                # Write serial number, action, and filename to info file
                info_file.write(f"{serial_number}\t{action_name}\t{str(filename).split('.')[0]+'.npy'}\n")
                
                process_video(video_path, holistic, output_folder)
                serial_number += 1

if __name__ == "__main__":
    input_folder = r"sliced_clips"  # Folder containing sliced video clips
    output_info_file = "dataset_info.txt"
    output_folder = C.save_dir
    process_video_clips(input_folder, output_info_file,output_folder)


libEGL warning: MESA-LOADER: failed to open zink: /usr/lib/dri/zink_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open zink: /usr/lib/dri/zink_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open zink: /usr/lib/dri/zink_dri.so: cannot open shared object fi